# INFERLIQ — a run from a cold checkout

This notebook takes the repository from a fresh clone to a completion printed
by the engine: it reports what the host has, installs the reference stack that
the comparison suite wants, builds `app_main` and the unit tests, runs both
suites, finds a checkpoint, and generates.

Almost none of that work lives here. `util/make.py` is the only build system
the project has, and every step below is one of its workflows — `install`,
`build`, `test`, `bench`, `clean` — run from the notebook rather than restated
in it. The engine itself has no dependencies at all; a C compiler is the whole
requirement, and the Python stack is only for the reference side.

Each cell carries its knobs as capitalised names at the top. Run the notebook
top to bottom, or set `MODEL` by hand and jump to the inference cell.

In [1]:
"""Find the checkout, load util/make.py, and set up the two helpers used below."""
import importlib.util
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path


def find_root(start=None):
    """The checkout is wherever util/make.py is, walking up from here."""
    here = Path(start or Path.cwd()).resolve()
    for folder in (here, *here.parents):
        if (folder / "util" / "make.py").exists():
            return folder
    raise RuntimeError("run this notebook from inside the inferliq checkout")


ROOT = find_root()
os.chdir(ROOT)

# util/make.py already knows which compilers this host has, where the build
# goes and what the reference stack needs, so the notebook loads it as a module
# instead of keeping a second copy of any of that.
_spec = importlib.util.spec_from_file_location("ill_make", ROOT / "util" / "make.py")
make = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(make)

BUILD = Path(make.BUILD)
SUFFIX = ".exe" if os.name == "nt" else ""
APP = BUILD / (make.APP + SUFFIX)


def shell(cmd, echo=True):
    """Runs a command with its output streaming into the cell; returns the exit code."""
    cmd = [str(part) for part in cmd]
    if echo:
        print("$ " + " ".join(cmd), flush=True)
    run = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in run.stdout:
        print(line, end="", flush=True)
    return run.wait()


def make_py(*args):
    """One util/make.py workflow, run with this notebook's interpreter."""
    return shell([sys.executable, ROOT / "util" / "make.py", *args])


def app(*args, quiet=True):
    """One captured run of build/app_main: (exit code, stdout, stderr).

    `quiet` suppresses the load progress, and with it the timing line the
    engine writes to stderr when a run finishes, so the inference cell below
    passes quiet=False.
    """
    cmd = [str(APP)] + [str(part) for part in args] + (["--quiet"] if quiet else [])
    done = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    return done.returncode, done.stdout, done.stderr


print(f"checkout    {ROOT}")
print(f"python      {platform.python_version()}  ({sys.executable})")
print(f"host        {platform.system()} {platform.machine()}, {os.cpu_count()} cores")

checkout    D:\inferliq
python      3.12.10  (d:\inferliq\.venv\Scripts\python.exe)
host        Windows AMD64, 4 cores


## 1. What the host has

A C compiler is the only thing the engine needs. `make.find_cc` is the same
probe the build uses, so what it reports here is what `make.py build` will
pick in a moment.

The reference stack — `torch`, `transformers`, `safetensors`, `numpy`,
`tokenizers` — is wanted by `test/test.py`, which is the comparison against
Hugging Face, and by the synthetic checkpoint further down. Nothing in the
engine's own path touches it.

In [ ]:
CC = make.find_cc(None)
print(f"compiler    {CC or 'none found — set CC, or pass --cc PATH to the build cell'}")
if CC:
    version = subprocess.run([CC, "--version"], capture_output=True, text=True)
    print(f"            {(version.stdout or version.stderr).splitlines()[0]}")

VENV = Path(make.VENV)
REFERENCE_PYTHON = make.venv_python()
_probe = subprocess.run(
    [REFERENCE_PYTHON, "-c",
     "import torch, transformers; print(f'torch {torch.__version__}, "
     "transformers {transformers.__version__}')"],
    capture_output=True, text=True)
HAVE_REFERENCE = _probe.returncode == 0

print(f"venv        {VENV if VENV.exists() else 'absent — the install cell makes one'}")
print(f"reference   {_probe.stdout.strip() if HAVE_REFERENCE else 'not installed: ' + ', '.join(make.NEEDS)}")

## 2. The reference stack

`python3 util/make.py install` makes `.venv` and puts the five packages in it.
It is a download of roughly two gigabytes, almost all of it `torch`, and it is
skipped outright when the stack already imports. Set `INSTALL_REFERENCE` to
`False` to go straight to the build: the engine is built and run without any of
it, and the test cell below then runs the C unit tests alone.

In [ ]:
INSTALL_REFERENCE = True        # False builds and runs the engine without it

if HAVE_REFERENCE:
    print("the reference stack is already importable; nothing to install")
elif not INSTALL_REFERENCE:
    print("skipped — set INSTALL_REFERENCE = True to fetch torch and transformers")
else:
    make_py("install")
    _probe = subprocess.run([make.venv_python(), "-c", "import torch, transformers"],
                            capture_output=True, text=True)
    HAVE_REFERENCE = _probe.returncode == 0
    REFERENCE_PYTHON = make.venv_python()
    print(f"\nreference stack {'ready' if HAVE_REFERENCE else 'still missing'}")

## 3. Build

`make.py build` compiles `app/main.c` into `build/app_main` and `test/test.c`
into `build/app_test`, each a single translation unit, after probing which
flags the compiler accepts. It takes a few seconds and reads nothing but the
source.

The flavours are alternatives to the default native build: `--portable` targets
a widely available instruction set rather than this host, `--no-simd` compiles
the scalar reference kernels, `--debug` turns optimisation off, and
`--sanitize` adds the address and undefined behaviour sanitizers.

In [ ]:
BUILD_FLAVOUR = []              # ["--portable"], ["--no-simd"], ["--debug"], ["--sanitize"]

code = make_py("build", *BUILD_FLAVOUR)
if code != 0:
    raise RuntimeError("build failed — the output above names the compiler and the flags")

print()
for leaf in sorted(BUILD.iterdir()):
    if leaf.is_file() and not leaf.name.endswith((".obj", ".pdb", ".ilk")):
        print(f"  {leaf.name:12} {leaf.stat().st_size / 1024:7.0f} KiB")

## 4. The two suites

`make.py test` runs `build/app_test` — unit checks over the engine internals —
and then `test/test.py`, which builds small Liquid checkpoints with random
weights, runs them through both this engine and `transformers`, and compares
logits position by position. That second half is skipped with a note when the
reference stack is absent, so this cell is useful either way.

`--no-checkpoint` holds back the suite that wants published weights; the
checkpoint is dealt with on its own terms two cells down, and running the
comparison against a 2.6B model here would cost minutes before the notebook
has generated anything.

In [ ]:
RUN_TESTS = True
TEST_FLAGS = ["--no-checkpoint"]        # drop this to add the checkpoint suite

if RUN_TESTS:
    code = make_py("test", *TEST_FLAGS)
    print("\nsuites pass" if code == 0 else f"\nthe test run exited {code}; see above")
else:
    print("skipped — set RUN_TESTS = True")

## 5. Which checkpoints are on disk

The published checkpoints ship in `ckpt/`, but their weights are stored with
Git LFS: a clone made without LFS leaves a short text stand-in in place of
every `.safetensors` shard and of `tokenizer.json`, and the engine reports the
folder as having no shards. The survey below reads the first bytes of each file
and says which of the two it is holding.

Only a causal checkpoint can be generated from — `Lfm2ForCausalLM` in the
architecture field. The encoder and the vision-language checkpoints in the same
folder are read by other things.

In [ ]:
MODEL = None            # set this by hand to a checkpoint folder to skip the search
SYNTHETIC = False       # flipped by the fallback cell below

CKPT = ROOT / "ckpt"
LFS_MARK = b"version https://git-lfs.github.com/spec/v1"


def is_pointer(path):
    """True when the file is a Git LFS stand-in rather than the bytes themselves."""
    with open(path, "rb") as fh:
        return fh.read(len(LFS_MARK)) == LFS_MARK


def survey(folder):
    """What one checkpoint folder holds, and whether its weights are really here."""
    config = folder / "config.json"
    if not config.exists() or is_pointer(config):
        return None
    facts = json.loads(config.read_text())
    shards = [s for s in sorted(folder.glob("*.safetensors"))
              if not s.name.startswith("adapter")]
    vocab = folder / "tokenizer.json"
    return {
        "name": folder.name,
        "arch": (facts.get("architectures") or ["unknown"])[0],
        "shards": len(shards),
        "waiting": sum(1 for s in shards if is_pointer(s)),
        "vocab": vocab.exists() and not is_pointer(vocab),
    }


FOUND = [row for row in (survey(f) for f in sorted(CKPT.iterdir()) if f.is_dir()) if row]

print(f"{'checkpoint':22}{'architecture':34}weights")
for row in FOUND:
    if not row["shards"]:
        state = "no shards in the folder"
    elif row["waiting"]:
        state = f"{row['waiting']} of {row['shards']} still Git LFS pointers"
    else:
        state = f"{row['shards']} shard(s) on disk"
    print(f"{row['name']:22}{row['arch']:34}{state}")

GENERATORS = [row for row in FOUND if row["arch"].endswith("ForCausalLM")]
READY = [row for row in GENERATORS if row["shards"] and not row["waiting"] and row["vocab"]]
if MODEL is None and READY:
    MODEL = CKPT / READY[0]["name"]

print()
print(f"model for this run: {MODEL if MODEL else 'none yet — the next two cells deal with that'}")

## 6. Fetching the weights

If the shards are still pointers, Git LFS fetches the bytes. It is a large
download — the 2.6B checkpoint is about five gigabytes — and it needs
[git-lfs](https://git-lfs.com) installed; where it is missing this cell prints
the two commands to run rather than pretending to do the work.

In [ ]:
FETCH_WEIGHTS = True            # False leaves the pointers alone

_have_lfs = subprocess.run(["git", "lfs", "version"], cwd=ROOT,
                           capture_output=True, text=True).returncode == 0

if MODEL is not None:
    print(f"{MODEL.name} is already on disk; nothing to fetch")
elif not GENERATORS:
    print("no causal checkpoint under ckpt/ to fetch")
elif not FETCH_WEIGHTS:
    print("skipped — set FETCH_WEIGHTS = True")
elif not _have_lfs:
    print("git-lfs is not installed. Install it from https://git-lfs.com, then:\n")
    print("    git lfs install")
    print(f"    git lfs pull --include \"ckpt/{GENERATORS[0]['name']}/**\"")
else:
    want = GENERATORS[0]["name"]
    shell(["git", "lfs", "install", "--local"])
    code = shell(["git", "lfs", "pull", "--include", f"ckpt/{want}/**"])
    row = survey(CKPT / want)
    if code == 0 and row and not row["waiting"]:
        MODEL = CKPT / want
        print(f"\n{want} is on disk")
    else:
        print(f"\nthe pull exited {code} and {want} is still incomplete")

## 7. A checkpoint that costs nothing

Where no published weights are on disk, the notebook can still run the engine
end to end on a small checkpoint built on the spot: six layers, a model
dimension of 32, random weights, and a byte level BPE trained over a handful of
strings. `test/test.py` already knows how to write both — it is what the parity
suite compares against `transformers` — so this cell calls that builder in the
reference interpreter rather than keeping a second copy of the shapes.

**The weights are random, so the text it generates means nothing.** What it
demonstrates is the path: a checkpoint folder read from disk, a prompt
tokenised, a forward pass per token, and pieces streamed back. Fetch the
published weights for output worth reading.

In [ ]:
SYNTHETIC_FALLBACK = True

_builder = """
import importlib.util, os, shutil, sys
spec = importlib.util.spec_from_file_location("ill_test", sys.argv[1])
suite = importlib.util.module_from_spec(spec)
spec.loader.exec_module(suite)

# The tokenizer is trained first because the vocabulary it settles on decides
# how wide the model's embedding has to be: a model narrower than its tokenizer
# is handed ids it has no row for, and the engine refuses the run.
room = sys.argv[2]
stage = os.path.join(room, "_vocab")
os.makedirs(stage, exist_ok=True)
vocab = suite.build_tokenizer(stage, "gpt2")

shape = dict(suite.SHAPES["hybrid_stack"])
shape["vocab_size"] = vocab.get_vocab_size()
folder, _ = suite.build_model(room, "synthetic", shape, seed=11)
shutil.move(os.path.join(stage, "tokenizer.json"), os.path.join(folder, "tokenizer.json"))
shutil.rmtree(stage, ignore_errors=True)
print(folder)
"""

if MODEL is not None:
    print(f"using {MODEL.name}; no synthetic checkpoint needed")
elif not SYNTHETIC_FALLBACK:
    print("skipped — set SYNTHETIC_FALLBACK = True")
elif not HAVE_REFERENCE:
    print("the synthetic checkpoint is written by transformers, which is not installed;\n"
          "run the install cell above, or fetch the published weights")
else:
    room = BUILD / "demo"
    shutil.rmtree(room / "synthetic", ignore_errors=True)
    room.mkdir(parents=True, exist_ok=True)
    done = subprocess.run([REFERENCE_PYTHON, "-c", _builder,
                           str(ROOT / "test" / "test.py"), str(room)],
                          cwd=ROOT, capture_output=True, text=True)
    if done.returncode != 0:
        print(done.stdout + done.stderr)
    else:
        MODEL = Path(done.stdout.strip().splitlines()[-1])
        SYNTHETIC = True
        print(f"built a synthetic checkpoint at {MODEL.relative_to(ROOT)}")

## 8. What the engine makes of it

`info` loads the checkpoint and reports what it found: the layer plan, the head
counts, the kernel width, how much the weights take resident, and which backend
and thread count the run would use. Nothing here is compiled in — the engine
reads all of it out of `config.json` and the shards, which is why it runs any
model of this family rather than one set of dimensions.

In [ ]:
if MODEL is None:
    print("no checkpoint available; fetch the weights or run the synthetic cell")
else:
    code, out, err = app("info", "--model", MODEL)
    print(out or err)

## 9. Inference

The run itself. `--temp 0` is greedy and reproducible; raise it, or set
`--top-p` and `--top-k`, to sample. `--quant q8` repacks the weights at load,
which halves the memory and roughly doubles decode at the cost of a few seconds
at startup. `--raw` skips the chat template and feeds the prompt verbatim,
which is what the synthetic checkpoint needs because it has no template.

The completion arrives on stdout and the load progress and timing line on
stderr, so the cell keeps them apart: the reply is displayed, and the rates are
read back out of the log beneath it.

In [ ]:
PROMPT = "Explain in three sentences why an inference engine can be written without dependencies."
SYSTEM = None                   # e.g. "Caveman mode: full."
MAX_TOKENS = 128
TEMPERATURE = 0.0               # 0 is greedy
QUANT = None                    # "q8" halves the memory and roughly doubles decode
THREADS = None                  # default is the host's core count

if MODEL is None:
    print("no checkpoint available; nothing to run")
else:
    args = ["generate", "--model", MODEL, "--prompt", PROMPT,
            "--max-tokens", MAX_TOKENS, "--temp", TEMPERATURE]
    if SYSTEM:
        args += ["--system", SYSTEM]
    if QUANT:
        args += ["--quant", QUANT]
    if THREADS:
        args += ["--threads", THREADS]
    if SYNTHETIC:
        args += ["--raw"]

    started = time.time()
    code, reply, log = app(*args, quiet=False)
    spent = time.time() - started

    if code != 0:
        print(log or reply)
    else:
        try:
            from IPython.display import Markdown, display
            display(Markdown(f"**{PROMPT}**\n\n---\n\n{reply.strip()}"))
        except ImportError:
            print(f"{PROMPT}\n\n{reply.strip()}")

        rates = re.search(r"prefill (\d+) tok in ([\d.]+)s, ([\d.]+) tok/s \| "
                          r"decode (\d+) tok in ([\d.]+)s, ([\d.]+) tok/s", log)
        print()
        if rates:
            read, fill_secs, fill_rate, made, step_secs, step_rate = rates.groups()
            print(f"prefill  {read} tok in {fill_secs}s — {fill_rate} tok/s")
            print(f"decode   {made} tok in {step_secs}s — {step_rate} tok/s")
        else:
            print(log.strip())
        print(f"wall     {spent:.2f}s, load included")
        if SYNTHETIC:
            print("\nrandom weights: the shape of the run is real, the text is not")

## 10. Throughput, and clearing up

`make.py bench` times prefill and decode on their own, which is the number to
quote rather than the wall time above — that one carries the load. Compare like
with like: the figure is a host's as much as the engine's, and a `q8` rate does
not sit beside a bf16 one.

`make.py clean` removes `build/`, and with it the binaries and anything the
synthetic cell wrote. The source tree carries nothing else the notebook made.

In [ ]:
RUN_BENCH = False               # a few minutes on a 2.6B checkpoint
CLEAN_AT_END = False            # removes build/, binaries and synthetic checkpoint alike

if RUN_BENCH and MODEL is not None:
    make_py("bench", "--model", MODEL, "--", "--batch", "256", "--max-tokens", "64")
elif RUN_BENCH:
    print("no checkpoint to bench")
else:
    print("bench skipped — set RUN_BENCH = True")

if CLEAN_AT_END:
    make_py("clean")

---

`README.md` is what the engine is and what the flags do, `GUIDE.md` the tour of
the implementation layer by layer, `TODO.md` what is still open, and
`CHANGES.md` the archive of every version with its numbers. `AGENTS.md`
describes how to work in the repository.